#### This noteboook reads a folder of documents and answer questions

##### LangChain for orchestration, Chroma for storage, and HuggingFace to run the Llama 3 model on HF's services of Inference API.

In [2]:
# All needed imports

# Data Science Libraries
from huggingface_hub import InferenceClient
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# Standard Libraries
import os

/root/opt/ra-i-g/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/tmp/ipykernel_2105912/555980068.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [8]:
# Remote LLM on HF
client = InferenceClient(token=os.getenv("HF_TOKEN"))
MODEL = "Qwen/Qwen2.5-7B-Instruct"

# Embeddings that turn the text chunks into lists of numbers
embedding_function = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2" # lightweight, open-source model that runs quickly on CPU, this model is only ~80MB
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 43050.65it/s]


In [4]:
# Loading the PDF document

def load_documents(folder_path: str):
    if not os.path.exists(folder_path):
        raise FileNotFoundError(f"Folder '{folder_path}' does not exist")

    documents = []
    for filename in os.listdir(folder_path):
        if filename.endswith(".pdf"):
            file_path = os.path.join(folder_path, filename)
            print(f"📄 Loading: {filename}")
            try:
                loader = PyPDFLoader(file_path)
                documents.extend(loader.load())
            except Exception as e:
                print(f"❌ Error loading {filename}: {e}")
    return documents

def split_text(documents):
    """
    Chunking the documents cause the LLMs have a context window limit, so we need to split the documents into smaller chunks
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200, # creates a sliding window, making sure we don’t lose context if a sentence is split between chunks
    )
    chunks = splitter.split_documents(documents)
    print(f"Created {len(chunks)} chunks")
    return chunks

In [10]:
def create_vector_store(chunks):
    """For Storing the embeddings in a vector store we can use Chroma, which allows us to perform similarity searches on the embeddings
    """
    vector_store = Chroma.from_documents(
        documents=chunks,
        embedding=embedding_function,
        persist_directory="../../data/train/rag/chroma_db", # saves the database in a folder called ./chroma_db. That way, you don’t have to rebuild the database every time you restart the app; it stays saved.
        collection_name="rag_docs"
    )
    return vector_store

In [11]:
def query_rag_system(query_text, vector_store):
    """
    Links the user, the database, and the LLM : looks at the users question and finds the top 3 most relevant chunks (k=3). 
    Then, it puts those chunks into a strict prompt.
    """
    # Retrieve top 3 relevant chunks
    retriever = vector_store.as_retriever(search_kwargs={"k": 3})
    docs = retriever.invoke(query_text)
    print(f"Docs ; {docs}")
    context = "\n\n".join(doc.page_content for doc in docs)

    # Generate answer via HF Inference API
    result = client.chat_completion(
        messages=[{
            "role": "user",
            "content": f"""You are a helpful assistant.
            Answer ONLY using the context below.
            If the answer is not present, say "I don't know."

            Context:
            {context}

            Question:
            {query_text}"""
                    }],
                    model=MODEL,
                    max_tokens=300
                )

    return result.choices[0].message.content.strip()


In [12]:
# Run the Pipeline

folder_path = "../../data/raw/pdf"  # Path to the folder containing PDF files

if not os.path.exists("../../data/train/rag/chroma_db"):
    print("No vector DB found. Creating one...")
    docs = load_documents(folder_path)
    chunks = split_text(docs)
    vector_store = create_vector_store(chunks)
    print("Vector database created")
else:
    print("Loading existing vector DB...")
    vector_store = Chroma(
        persist_directory="../../data/train/rag/chroma_db",
        embedding_function=embedding_function,
        collection_name="rag_docs"
    )

while True:
    query = input("\nAsk a question (or type 'exit'): ")
    if query.lower() == "exit":
        break

    print("Thinking...")
    answer = query_rag_system(query, vector_store)
    print("\n Answer:\n", answer)

No vector DB found. Creating one...
📄 Loading: Full-49.pdf
📄 Loading: Full-47.pdf
📄 Loading: Full-48.pdf
Created 7 chunks
Vector database created
Thinking...
Docs ; [Document(id='00c895bf-fde6-4a3a-aacc-75066ec66d95', metadata={'source': '../../data/raw/pdf/Full-49.pdf', 'total_pages': 1, 'moddate': '2026-06-25T11:24:39+00:00', 'page': 0, 'creationdate': '', 'creator': 'PyPDF', 'producer': 'iLovePDF', 'page_label': '1'}, page_content='Sun. These are solar prominences and surround the ecli pse Sun you will see a whitish extension, the corona or Sun’s outer\natmosphere.\nEclipse: Lore and Legend\nThere are some interesting eclipse stories over the passage of civilization. For example, was Stonehenge a system to predict lunar\neclipses or something else? In China, the term for a n eclipse is chih, which also means to eat; something wa s eating the Sun. The\nancient Chaldeans believed that an eclipse was a display of the Moon’s anger. The Babylonians determined the eclipse’s “quadrant”\n– 